<a href="https://colab.research.google.com/github/mdzubayereashraf/Ostad-AI-Engineering-Course-Assignments/blob/main/Assignmen_No_19(Fine_Tuning_Transformers_for_Question_Answering_%7C%7C_Text_Classification_%26_Sentiment_Analysis_using_Hugging_Face).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction**

Natural Language Processing (NLP) is a branch of Artificial Intelligence that enables computers to understand, interpret, and process human language. In this assignment, I will fine-tune a pre-trained BERT (Bidirectional Encoder Representations from Transformers) model for the Question Answering (QA) task using the Stanford Question Answering Dataset (SQuAD v1.1) and the Hugging Face Transformers library.

### **Text Classification vs. Question Answering:**

Text Classification and Question Answering are two important NLP tasks, but they serve different purposes.

- **Text Classification:** It involves assigning predefined categories or labels to a given text. For example, classifying a movie review as positive or negative is a sentiment analysis task.

- **Question Answering:** It involves finding a relevant answer to a question based on a given context. For example, if the context describes Albert Einstein's achievements and the question asks who developed the theory of relativity, the model should identify "Albert Einstein" as the answer.

### **Objective:**

The main objective of this assignment is to fine-tune the pre-trained BERT model on the SQuAD v1.1 dataset so that it can understand questions and identify relevant answers from given passages.

I will use the Hugging Face Transformers library for tokenization and model training, and evaluate the model's performance using Exact Match (EM) and F1 Score.

### **Install and import libraries**

In [1]:
!pip install transformers datasets evaluate accelerate

In [2]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

import evaluate

### **Load the SQuAD Dataset**

In [4]:
dataset = load_dataset("rajpurkar/squad")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

### **Explore dataset**

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [6]:
print(dataset["train"][0])

{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [7]:
sample = dataset["train"][0]

print("Question:", sample["question"])

print("\nContext:", sample["context"])

print("\nAnswer:", sample["answers"])

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}
